# 10 · Eavesdropper demo — why BB84 is secure

BB84's security claim is that **eavesdropping is detectable**: an attacker who measures
photons in transit disturbs them, raising the quantum bit error rate (QBER). This
notebook makes that concrete with an **intercept-resend** attack.

For a fraction `f` of photons, Eve measures in a randomly chosen basis (she doesn't know
Alice's) and resends a fresh photon prepared from her result. On the *sifted* key:

- Eve guesses the right basis (prob ½): she learns and resends the bit correctly → no error.
- Eve guesses wrong (prob ½): she resends in the wrong basis, so Bob's outcome is random → error with prob ½.

So a fully-tapped channel adds **QBER = ½ · ½ = 0.25**, and tapping a fraction `f` adds
**≈ 0.25·f** on top of the channel's intrinsic QBER. Past the **~11%** Shor–Preskill
threshold the secure key rate is zero and the parties abort.

**Runs entirely locally — no FABRIC slice needed.** It exercises the real `qne/eve.py`,
detector, and `BB84Protocol.secure_key_fraction`; the same Eve is wired into the
distributed path via `node_runner --eve-fraction` (bridged in §5).

## 1 · A BB84 run with an eavesdropper

In [ ]:
import sys, types
from pathlib import Path
import numpy as np

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))

from qne.eve import InterceptResendEve, expected_sifted_qber
from qne.detector import Detector
from qne.bb84 import AliceRecord, BobRecord, BB84Protocol


def run_bb84_with_eve(f, n=20000, fidelity=1.0, efficiency=1.0,
                      sample_fraction=0.2, seed=0):
    """One prepare-and-measure BB84 run through an intercept-resend Eve.

    fidelity=1.0 isolates Eve (no channel noise); lower F adds intrinsic QBER (1-F)/2.
    Returns measured QBER, Shor-Preskill secure fraction, and sifted-key size."""
    rng = np.random.default_rng(seed)
    eve = InterceptResendEve(intercept_fraction=f, seed=seed + 1)
    det = Detector(efficiency=efficiency, dark_count_rate=0.0,
                   polarization_error=1.0 - fidelity, seed=seed + 2)
    alice, bob = [], []
    for i in range(n):
        a_basis, a_bit = int(rng.integers(0, 2)), int(rng.integers(0, 2))
        e_basis, e_bit = eve.intercept(a_basis, a_bit)          # Eve taps in transit
        ev = det.detect(types.SimpleNamespace(basis=e_basis, state=e_bit,
                                              sequence_num=i))
        if ev.detected:
            alice.append(AliceRecord(i, a_basis, a_bit))        # Alice announces TRUE basis
            bob.append(BobRecord(i, ev.basis, ev.bit_value))
    proto = BB84Protocol(sample_fraction=sample_fraction, seed=seed + 3)
    sifted = proto.sift(alice, bob)
    qest = proto.estimate_qber(sifted)
    return {'f': f, 'qber': qest.qber,
            'secure_fraction': BB84Protocol.secure_key_fraction(qest.qber),
            'sifted': sifted.sifted_count,
            'intercepted': eve.photons_intercepted}


demo = run_bb84_with_eve(f=1.0, seed=0)
print(f"Fully-tapped channel (f=1): QBER={demo['qber']:.3f} "
      f"(theory {expected_sifted_qber(1.0):.3f}), "
      f"secure fraction={demo['secure_fraction']:.3f}")

## 2 · Sweep Eve's tap fraction

Vary `f` from 0 (no Eve) to 1 (every photon tapped), with repetitions for error bars.
We run two channels: an **ideal** one (F=1, so QBER is *pure Eve*) and a **realistic**
one (F=0.98, ~1% intrinsic QBER) so you can see Eve's disturbance stack on top.

In [ ]:
FRACTIONS = np.linspace(0, 1, 11)
REPS = 5

def sweep(fidelity):
    rows = []
    for f in FRACTIONS:
        runs = [run_bb84_with_eve(f, fidelity=fidelity, seed=100 * r + 7)
                for r in range(REPS)]
        q = np.array([r['qber'] for r in runs])
        s = np.array([r['secure_fraction'] for r in runs])
        rows.append({'f': f, 'qber': q.mean(), 'qber_std': q.std(),
                     'secure': s.mean()})
    return rows

ideal = sweep(fidelity=1.0)
noisy = sweep(fidelity=0.98)

print(f"{'f':>5} {'QBER(F=1)':>10} {'secure':>8}   {'QBER(F=.98)':>12} {'secure':>8}")
for a, b in zip(ideal, noisy):
    print(f"{a['f']:>5.2f} {a['qber']:>10.4f} {a['secure']:>8.3f}   "
          f"{b['qber']:>12.4f} {b['secure']:>8.3f}")

## 3 · The security picture

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
fig.suptitle('Intercept-resend eavesdropper on BB84', fontsize=13, weight='bold')

fs = [r['f'] for r in ideal]
ax1.errorbar(fs, [r['qber'] for r in ideal], yerr=[r['qber_std'] for r in ideal],
             marker='o', label='measured, F=1.0 (pure Eve)')
ax1.errorbar(fs, [r['qber'] for r in noisy], yerr=[r['qber_std'] for r in noisy],
             marker='s', label='measured, F=0.98')
ax1.plot(fs, [expected_sifted_qber(f) for f in fs], 'k--', lw=1, label='theory 0.25·f')
ax1.axhline(0.11, color='crimson', ls=':', lw=1.5, label='~11% security threshold')
ax1.set(xlabel="Eve's tap fraction  f", ylabel='sifted QBER',
        title='Eavesdropping raises the error rate')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

ax2.plot(fs, [r['secure'] for r in ideal], marker='o', label='F=1.0')
ax2.plot(fs, [r['secure'] for r in noisy], marker='s', label='F=0.98')
ax2.set(xlabel="Eve's tap fraction  f", ylabel='secure key fraction',
        title='...and the secure key rate collapses')
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

fig.tight_layout()
out = PROJECT_DIR / 'paper' / 'figures' / 'eve_sweep.png'
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, dpi=150, bbox_inches='tight')
print(f'saved {out}')
plt.show()

Read the left panel: with no channel noise (F=1) the measured QBER tracks the `0.25·f`
theory line exactly. On the realistic channel (F=0.98) it sits ~1% higher — Eve's
disturbance **adds to** the intrinsic error. The right panel shows the consequence: once
QBER crosses ~11%, the secure fraction is zero and the key is refused. Note the noisy
channel crosses the threshold at a **smaller** `f` — less headroom for an eavesdropper.

## 5 · Bridge to the distributed system (optional)

The same Eve is wired into the two-process BB84 path. This cell launches Bob + Alice on
localhost at `f=0` and `f=1` and shows the identical effect on the real protocol. It
needs the `qne-sequence` package importable; if anything fails it prints SKIP and the
in-process results above still stand.

In [ ]:
import json, os, subprocess, time

def run_distributed(f, port):
    env = dict(os.environ)
    env['PYTHONPATH'] = f"{PROJECT_DIR / 'qne-sequence'}{os.pathsep}{PROJECT_DIR}"
    def spawn(role, peer):
        return subprocess.Popen(
            [sys.executable, '-m', 'qne_sequence.node_runner', '--role', role,
             '--name', role, '--peer', peer, '--host', '127.0.0.1', '--port', str(port),
             '--num-pulses', '12000', '--fidelity', '1.0', '--sample-fraction', '0.2',
             '--seed', '42', '--eve-fraction', str(f)],
            cwd=str(PROJECT_DIR / 'qne-sequence'), env=env,
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    bob = spawn('bob', 'alice'); time.sleep(1); alice = spawn('alice', 'bob')
    out, _ = alice.communicate(timeout=90); bob.communicate(timeout=30)
    line = next(ln for ln in out.splitlines() if ln.startswith('{'))
    return json.loads(line)

try:
    for f, port in [(0.0, 57601), (1.0, 57602)]:
        r = run_distributed(f, port)
        print(f"distributed  f={f}: QBER={r['qber']:.3f}  "
              f"secure_fraction={r['secure_fraction']:.3f}  "
              f"eve_intercepted={r.get('eve_photons_intercepted')}")
except Exception as e:
    print(f'SKIP distributed bridge ({type(e).__name__}: {e}) — in-process results above stand.')

## 6 · Verify

In [ ]:
checks = []
def rec(name, ok, detail=''):
    checks.append(ok); print(f"  [{'PASS' if ok else 'FAIL'}] {name}"
                             + (f' — {detail}' if detail else ''))

q1 = ideal[-1]['qber']      # f = 1.0, F = 1.0
q0 = ideal[0]['qber']       # f = 0.0
rec('no Eve -> clean channel', q0 < 0.02, f'QBER(f=0)={q0:.4f}')
rec('full intercept -> ~25% QBER', abs(q1 - 0.25) < 0.02, f'QBER(f=1)={q1:.4f}')
rec('QBER rises monotonically with f',
    all(ideal[i]['qber'] <= ideal[i+1]['qber'] + 0.02 for i in range(len(ideal)-1)))
rec('full intercept breaks security', ideal[-1]['secure'] == 0.0)
rec('secure fraction is intact with no Eve', ideal[0]['secure'] > 0.9)

print('\nALL CHECKS PASSED' if checks and all(checks)
      else '\nSOME CHECKS FAILED')

## Takeaways

- **Eavesdropping is not silent.** Measuring photons in transit costs the attacker
  information they can't hide — it shows up as QBER, linear in how much they tap.
- **The ~11% threshold is the whole game.** Below it, error correction + privacy
  amplification can distill a secure key; above it, no secure key exists and BB84 aborts.
- This is why QKD's security is *physical*, not computational: there is no eavesdropping
  strategy that beats the disturbance, regardless of the attacker's computing power.

**Next:** a beam-splitting / PNS attacker (which decoy states defend against — see
`qne/decoy.py`), and running this sweep over a real FABRIC link to see whether WAN
conditions change the detectability picture.